1.Train a simple neural network to classify whether a review is positive or negative using a small dataset (you can use any dataset of your choice), then save the trained model to a .h5 file using model.save('sentiment_model.h5').

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Sample Dataset
reviews = [
    "This movie is amazing",
    "I love this product",
    "Excellent service",
    "Very happy with the purchase",
    "Fantastic experience",
    "This movie is terrible",
    "Worst product ever",
    "I hate this",
    "Very disappointing",
    "Bad experience"
]

labels = [1,1,1,1,1,0,0,0,0,0]  # 1=Positive, 0=Negative

# Tokenization
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(reviews)

sequences = tokenizer.texts_to_sequences(reviews)
X = pad_sequences(sequences, maxlen=5)

y = np.array(labels)

# Build Model
model = Sequential([
    Dense(16, activation='relu', input_shape=(5,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train Model
model.fit(X, y, epochs=20, validation_split=0.2)

# Save Model
model.save('sentiment_model.h5')

print("Model saved as sentiment_model.h5")

2.Write code to load the 'sentiment_model.h5' file you saved and use it to predict the sentiment of three new sample reviews.

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load model
loaded_model = load_model('sentiment_model.h5')

# New Reviews
new_reviews = [
    "I really enjoyed this movie",
    "Worst service ever",
    "The product is good"
]

# Convert to sequences
new_seq = tokenizer.texts_to_sequences(new_reviews)
new_pad = pad_sequences(new_seq, maxlen=5)

# Predictions
predictions = loaded_model.predict(new_pad)

for review, pred in zip(new_reviews, predictions):
    sentiment = "Positive" if pred > 0.5 else "Negative"
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}\n")

3.Implement model checkpointing during training so that the model's weights are saved every time the validation accuracy improves.<br><br><em><strong>Hint:</strong> Use the ModelCheckpoint callback in Keras with save_best_only=True.</em>

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

model.fit(
    X,
    y,
    epochs=20,
    validation_split=0.2,
    callbacks=[checkpoint]
)

4.Export your trained model in a format suitable for deployment (such as TensorFlow SavedModel format), and explain in one line how this format helps with deploying the model to a web or mobile app

In [ ]:
model.export("saved_sentiment_model")

In [ ]:
model_json = model.to_json()

with open("model_architecture.json", "w") as json_file:
    json_file.write(model_json)

print("Architecture saved")

model.save_weights("model_weights.weights.h5")

print("Weights saved")

from tensorflow.keras.models import model_from_json

# Load Architecture
with open("model_architecture.json", "r") as json_file:
    loaded_json = json_file.read()

loaded_model = model_from_json(loaded_json)

# Load Weights
loaded_model.load_weights("model_weights.weights.h5")

# Compile Model
loaded_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model loaded successfully")

In [ ]:
original_pred = model.predict(new_pad)
loaded_pred = loaded_model.predict(new_pad)

print("Original Predictions:")
print(original_pred)

print("\nLoaded Model Predictions:")
print(loaded_pred)